## 

First attempts in numrisk/fmri_analysis/surface/vis_surface_....

In [1]:
from nilearn import image
from nilearn import surface
import os.path as op
import numpy as np
import nibabel as nib

bids_folder_freesurfer = '/mnt_03/ds-dnumrisk'
bids_folder_data = '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk'


In [2]:
import cortex

In [ ]:
sub = '01' #'average'
confspec = '36Pscrub3BPfilter'
task = 'magjudge'
ses = 1

space = 'fsnative' #'fsaverage'
dataset = 'dnumrisk'

pyc_subject = space if space == 'fsaverage' else f'{dataset}.sub-{sub}'

source_folder = op.join(bids_folder_data,'derivatives','networks_infomap_full', f'sub-{sub}')

surf_L = surface.load_surf_data(op.join(source_folder, f'sub-{sub}_ses-{ses}_task-{task}_space-{space}_hemi-L_consensus_labels.surf.gii'))
surf_R = surface.load_surf_data(op.join(source_folder, f'sub-{sub}_ses-{ses}_task-{task}_space-{space}_hemi-R_consensus_labels.surf.gii'))

surf_map_fsnative = np.concatenate([surf_L, surf_R])
print(surf_map_fsnative.shape)
ds_1 = cortex.Vertex(surf_map_fsnative, pyc_subject)
cortex.webshow(ds_1, port=8000)  

# ssh -L 8080:localhost:8000 sciencecloud

In [ ]:
sub = '33' #'average'
confspec = '36Pscrub3BPfilter'
task = 'magjudge'
ses = 1

space = 'fsnative' #'fsaverage'
dataset = 'dnumrisk'

pyc_subject = space if space == 'fsaverage' else f'{dataset}.sub-{sub}'

source_folder = op.join(bids_folder_data,'derivatives','networks_infomap_full', f'sub-{sub}')

surf_L = surface.load_surf_data(op.join(source_folder, f'sub-{sub}_ses-{ses}_task-{task}_space-{space}_hemi-L_consensus_labels.surf.gii'))
surf_R = surface.load_surf_data(op.join(source_folder, f'sub-{sub}_ses-{ses}_task-{task}_space-{space}_hemi-R_consensus_labels.surf.gii'))

surf_map_fsnative = np.concatenate([surf_L, surf_R])
print(surf_map_fsnative.shape)
ds_1 = cortex.Vertex(surf_map_fsnative, pyc_subject)
cortex.webshow(ds_1, port=8001)  

# ssh -L 8080:localhost:8000 sciencecloud

(253792,)
Started server on port 8001


<JS: window.viewer>

In [ ]:
# Both: gradients and network labels

sub = '10' #'average'
task = 'magjudge'
ses = 1

space = 'fsnative' #'fsaverage'
dataset = 'dnumrisk'

pyc_subject = space if space == 'fsaverage' else f'{dataset}.sub-{sub}'

source_folder = op.join(bids_folder_data,'derivatives','networks_infomap_full', f'sub-{sub}')
surf_L = surface.load_surf_data(op.join(source_folder, f'sub-{sub}_ses-{ses}_task-{task}_space-{space}_hemi-L_consensus_labels.surf.gii'))
surf_R = surface.load_surf_data(op.join(source_folder, f'sub-{sub}_ses-{ses}_task-{task}_space-{space}_hemi-R_consensus_labels.surf.gii'))
surf_map_fsnative_1 = np.concatenate([surf_L, surf_R])

grad_n = 2
source_folder = op.join(bids_folder_data,'derivatives','gradients.36Pscrub3BPfilterrunFD104', f'sub-{sub}')
surf_L = surface.load_surf_data(op.join(source_folder, f'sub-{sub}_ses-{ses}_task-{task}_space-{space}_hemi-L_grad-{grad_n}.surf.gii'))
surf_R = surface.load_surf_data(op.join(source_folder, f'sub-{sub}_ses-{ses}_task-{task}_space-{space}_hemi-R_grad-{grad_n}.surf.gii'))
surf_map_fsnative_2 = np.concatenate([surf_L, surf_R])

v1 = cortex.Vertex(surf_map_fsnative_1, pyc_subject)
v2 = cortex.Vertex(surf_map_fsnative_2, pyc_subject)

ds = cortex.Dataset(
    networks=v1,
    gradient=v2
)
cortex.webshow(ds, port=8001)

# ssh -L 8080:localhost:8000 sciencecloud

Generating new ctm file...
wm
wm
inflated
inflated
Started server on port 8001


<JS: window.viewer>

Stopping server


## Transform surfaces

In [2]:
from numrisk.fmri_analysis.surface.utils import transform_surfaces

250929-13:13:51,527 nipype.utils WARNING:
	 A newer version (1.10.0) of nipy/nipype is available. You are using 1.8.6


In [3]:

sub = '10' #'average'
confspec = '36Pscrub3BPfilter'
task = 'magjudge'
ses = 1
#bids_folder_data = '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk'

target_folder = op.join(bids_folder_data,'derivatives','networks_infomap_full_01')
fn_consens_mapping = op.join(target_folder, f'sub-{sub}_consensusMapping_confspec-{confspec}.npy')
consensus_labels = np.load(fn_consens_mapping)
np.shape(consensus_labels)

from numrisk.fmri_analysis.gradients.utils import get_basic_mask
mask, labeling_noParcel = get_basic_mask()

modules_fsav5 = np.full(mask.shape[0], np.nan, dtype=float)
modules_fsav5[mask] = consensus_labels
np.shape(modules_fsav5)

[get_dataset_dir] Dataset found in /home/ubuntu/nilearn_data/destrieux_surface


(20484,)

In [4]:
source_space = 'fsaverage5'
target_space= 'fsnative' #'fsaverage' # 
#bids_folder_freesurfer = '/mnt_03/ds-dnumrisk'

target_folder = op.join(bids_folder_data,'derivatives','networks_infomap_full', f'sub-{sub}')
target_space=f'sub-{sub}' if target_space=='fsnative' else target_space

import os
os.makedirs(target_folder) if not op.exists(target_folder) else None

gm_both = np.split(modules_fsav5,2) # for i, hemi in enumerate(['L', 'R']): --> left first
for i, hemi in enumerate(['L', 'R']):
    gii_im_datar = nib.gifti.gifti.GiftiDataArray(data=gm_both[i].astype(np.float32))
    gii_im = nib.gifti.gifti.GiftiImage(darrays= [gii_im_datar])

    out_file = op.join(target_folder, f'sub-{sub}_ses-{ses}_task-{task}_space-{source_space}_hemi-{hemi}_consensus_labels.surf.gii')
    gii_im.to_filename(out_file) 

    fs_hemi = 'lh' if hemi=='L' else 'rh'
    transform_surfaces(in_file=out_file, fs_hemi=fs_hemi, bids_folder=bids_folder_freesurfer,
                            source_space= source_space, target_space=target_space)

250929-13:11:04,669 nipype.interface INFO:
	 stderr 2025-09-29T13:11:04.669766:** DA[0] has coordsys with intent NIFTI_INTENT_NONE (should be NIFTI_INTENT_POINTSET)
250929-13:11:05,651 nipype.interface INFO:
	 stdout 2025-09-29T13:11:05.651191:
250929-13:11:05,652 nipype.interface INFO:
	 stdout 2025-09-29T13:11:05.651191:7.3.2
250929-13:11:05,652 nipype.interface INFO:
	 stdout 2025-09-29T13:11:05.651191:
250929-13:11:05,652 nipype.interface INFO:
	 stdout 2025-09-29T13:11:05.651191:setenv SUBJECTS_DIR /mnt_03/ds-dnumrisk/derivatives/freesurfer
250929-13:11:05,653 nipype.interface INFO:
	 stdout 2025-09-29T13:11:05.651191:cd /home/ubuntu/git/parietal_patterns/networks_indTopology
250929-13:11:05,653 nipype.interface INFO:
	 stdout 2025-09-29T13:11:05.651191:mri_surf2surf --hemi lh --tval /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk/derivatives/networks_infomap_full/sub-10/sub-10_ses-1_task-magjudge_space-fsnative_hemi-L_consensus_labels.surf.gii --sval /mnt_AdaBD_largefi

In [4]:
# Gradient maps
#sub = '33'

grad_n = 2 # -1 for indexing numpy array ;) ! 
ses = 1
task = 'magjudge'
source_space = 'fsaverage5' #'fsaverage5' misspelled before!
target_space= 'fsnative' #'fsaverage' # 
bids_folder_freesurfer = '/mnt_03/ds-dnumrisk'

target_folder = op.join(bids_folder_data,'derivatives','gradients.36Pscrub3BPfilterrunFD104', f'sub-{sub}')
target_space=f'sub-{sub}' if target_space=='fsnative' else target_space

gms = np.load(op.join(target_folder, f'sub-{sub}_g-aligned_space-fsaverag5_n10.npy')) # misspelled before!!
gm_both = np.split(gms[grad_n-1],2) # for i, hemi in enumerate(['L', 'R']): --> left first
for i, hemi in enumerate(['L', 'R']):
    gii_im_datar = nib.gifti.gifti.GiftiDataArray(data=gm_both[i].astype(np.float32))
    gii_im = nib.gifti.gifti.GiftiImage(darrays= [gii_im_datar])

    out_file = op.join(target_folder, f'sub-{sub}_ses-{ses}_task-{task}_space-{source_space}_hemi-{hemi}_grad-{grad_n}.surf.gii')
    gii_im.to_filename(out_file) 

    fs_hemi = 'lh' if hemi=='L' else 'rh'
    transform_surfaces(in_file=out_file, fs_hemi=fs_hemi, bids_folder=bids_folder_freesurfer,
                            source_space= source_space, target_space=target_space)

250929-13:14:05,614 nipype.interface INFO:
	 stderr 2025-09-29T13:14:05.614346:** DA[0] has coordsys with intent NIFTI_INTENT_NONE (should be NIFTI_INTENT_POINTSET)
250929-13:14:06,482 nipype.interface INFO:
	 stdout 2025-09-29T13:14:06.482553:
250929-13:14:06,483 nipype.interface INFO:
	 stdout 2025-09-29T13:14:06.482553:7.3.2
250929-13:14:06,483 nipype.interface INFO:
	 stdout 2025-09-29T13:14:06.482553:
250929-13:14:06,484 nipype.interface INFO:
	 stdout 2025-09-29T13:14:06.482553:setenv SUBJECTS_DIR /mnt_03/ds-dnumrisk/derivatives/freesurfer
250929-13:14:06,484 nipype.interface INFO:
	 stdout 2025-09-29T13:14:06.482553:cd /home/ubuntu/git/parietal_patterns/networks_indTopology
250929-13:14:06,484 nipype.interface INFO:
	 stdout 2025-09-29T13:14:06.482553:mri_surf2surf --hemi lh --tval /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk/derivatives/gradients.36Pscrub3BPfilterrunFD104/sub-10/sub-10_ses-1_task-magjudge_space-fsnative_hemi-L_grad-2.surf.gii --sval /mnt_AdaBD_lar